# Notebook 0: Exploratory Data Analysis & Statistical Analysis
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

This notebook performs comprehensive EDA and statistical analysis on the stock price data:
- TLKM, BBCA, ASII, UNVR (Daily, Weekly, Monthly, Yearly)

**Contents:**
1. Data Loading & Overview
2. Descriptive Statistics
3. Data Distribution Analysis
4. Normality Tests (Shapiro-Wilk, Kolmogorov-Smirnov, D'Agostino)
5. Stationarity Tests (Augmented Dickey-Fuller)
6. Correlation Analysis
7. Time Series Visualization
8. Returns Analysis


In [4]:
# ============================================================
# 1. IMPORTS & SETUP
# ============================================================
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add notebook directory to path
sys.path.insert(0, os.path.dirname(os.path.abspath('stock_prediction_utils.py')))
sys.path.insert(0, '.')

from stock_prediction_utils import *

set_seed()
set_ieee_style()

# DATA DIRECTORY - Update this path to where your data files are
DATA_DIR = 'dataset'  # Points to dataset folder with all stock data files

# Create output directories
os.makedirs('figures/eda', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("Setup complete!")

Setup complete!


## 1. Data Loading & Overview

In [7]:
# ============================================================
# LOAD ALL DATA
# ============================================================
print("Loading Daily Data...")
daily_data = {}
for stock in STOCKS:
    filepath = get_data_paths(DATA_DIR, stock, 'daily')
    daily_data[stock] = load_stock_data(filepath)

print("\nLoading Weekly Data...")
weekly_data = {}
for stock in STOCKS:
    filepath = get_data_paths(DATA_DIR, stock, 'weekly')
    weekly_data[stock] = load_stock_data(filepath)

print("\nLoading Monthly Data...")
monthly_data = {}
for stock in STOCKS:
    filepath = get_data_paths(DATA_DIR, stock, 'monthly')
    monthly_data[stock] = load_stock_data(filepath)

print("\nLoading Yearly Data...")
yearly_data = {}
for stock in STOCKS:
    filepath = get_data_paths(DATA_DIR, stock, 'yearly')
    yearly_data[stock] = load_stock_data(filepath)

print("\nAll data loaded successfully!")

Loading Daily Data...

Loading Weekly Data...

Loading Monthly Data...

Loading Yearly Data...

All data loaded successfully!


In [8]:
# Data shape overview
print("=" * 70)
print(f"{'Stock':<8} {'Daily':<12} {'Weekly':<12} {'Monthly':<12} {'Yearly':<12}")
print("=" * 70)
for stock in STOCKS:
    print(f"{stock:<8} {len(daily_data[stock]):<12} {len(weekly_data[stock]):<12} "
          f"{len(monthly_data[stock]):<12} {len(yearly_data[stock]):<12}")

print("\n--- First 5 rows of each daily dataset ---")
for stock in STOCKS:
    print(f"\n{stock} Daily Data:")
    print(daily_data[stock].head())

print("\n--- Last 5 rows of each daily dataset ---")
for stock in STOCKS:
    print(f"\n{stock} Daily Data:")
    print(daily_data[stock].tail())

Stock    Daily        Weekly       Monthly      Yearly      
TLKM     5243         1102         256          22          
BBCA     5244         1102         256          22          
ASII     5244         1102         256          22          
UNVR     5245         1102         256          22          

--- First 5 rows of each daily dataset ---

TLKM Daily Data:
        Date       Close
0 2004-09-28  341.332367
1 2004-09-29  341.332367
2 2004-09-30  343.401062
3 2004-10-01  347.538422
4 2004-10-04  359.950562

BBCA Daily Data:
        Date       Close
0 2004-09-28  125.228340
1 2004-09-29  126.813507
2 2004-09-30  126.813507
3 2004-10-01  128.398712
4 2004-10-04  134.739349

ASII Daily Data:
        Date       Close
0 2004-09-28  273.852600
1 2004-09-29  277.970673
2 2004-09-30  282.088715
3 2004-10-01  288.265808
4 2004-10-04  306.797241

UNVR Daily Data:
        Date       Close
0 2004-09-28  331.884369
1 2004-09-29  326.855804
2 2004-09-30  326.855804
3 2004-10-01  331.884369
4 20

## 2. Descriptive Statistics

In [9]:
# ============================================================
# DESCRIPTIVE STATISTICS - DAILY DATA
# ============================================================
desc_stats = []
for stock in STOCKS:
    stat = compute_descriptive_stats(daily_data[stock], stock)
    desc_stats.append(stat)

desc_df = pd.DataFrame(desc_stats)
print("\n" + "=" * 80)
print("  DESCRIPTIVE STATISTICS - Daily Close Prices")
print("=" * 80)
print(desc_df.to_string(index=False))

# Save to CSV
desc_df.to_csv('results/descriptive_statistics_daily.csv', index=False)
print("\nSaved to results/descriptive_statistics_daily.csv")


  DESCRIPTIVE STATISTICS - Daily Close Prices
Stock  Count      Mean    Median   Std Dev     Variance      Min        Max      Range  Skewness  Kurtosis  Q1 (25%)  Q3 (75%)       IQR  CV (%)
 TLKM   5243 1847.7168 1764.6804 1037.1583 1075697.3848 341.3324  3982.0615  3640.7292    0.1525   -1.4608  827.7448 2740.7927 1913.0479 56.1319
 BBCA   5244 3276.1477 2155.7439 2958.0092 8749818.1834 125.2283 10500.9971 10375.7687    0.7847   -0.7214  733.3437 5450.4156 4717.0720 90.2892
 ASII   5244 3382.5800 3999.3096 1690.1827 2856717.6980 273.8526  6725.0000  6451.1474   -0.6439   -0.9488 1760.7285 4603.9990 2843.2705 49.9673
 UNVR   5245 3484.3656 3233.7654 2328.8251 5423426.4490 314.2844  8418.5703  8104.2859    0.3013   -1.2057 1362.0469 5667.8247 4305.7778 66.8364

Saved to results/descriptive_statistics_daily.csv


In [10]:
# Descriptive statistics for ALL timeframes
all_desc = []
for tf_name, tf_data in [('Daily', daily_data), ('Weekly', weekly_data), 
                           ('Monthly', monthly_data), ('Yearly', yearly_data)]:
    for stock in STOCKS:
        stat = compute_descriptive_stats(tf_data[stock], stock)
        stat['Timeframe'] = tf_name
        all_desc.append(stat)

all_desc_df = pd.DataFrame(all_desc)
cols = ['Timeframe', 'Stock'] + [c for c in all_desc_df.columns if c not in ['Timeframe', 'Stock']]
all_desc_df = all_desc_df[cols]
print("\nAll Timeframes - Descriptive Statistics:")
print(all_desc_df.to_string(index=False))
all_desc_df.to_csv('results/descriptive_statistics_all_timeframes.csv', index=False)


All Timeframes - Descriptive Statistics:
Timeframe Stock  Count      Mean    Median   Std Dev     Variance      Min        Max      Range  Skewness  Kurtosis  Q1 (25%)  Q3 (75%)       IQR  CV (%)
    Daily  TLKM   5243 1847.7168 1764.6804 1037.1583 1075697.3848 341.3324  3982.0615  3640.7292    0.1525   -1.4608  827.7448 2740.7927 1913.0479 56.1319
    Daily  BBCA   5244 3276.1477 2155.7439 2958.0092 8749818.1834 125.2283 10500.9971 10375.7687    0.7847   -0.7214  733.3437 5450.4156 4717.0720 90.2892
    Daily  ASII   5244 3382.5800 3999.3096 1690.1827 2856717.6980 273.8526  6725.0000  6451.1474   -0.6439   -0.9488 1760.7285 4603.9990 2843.2705 49.9673
    Daily  UNVR   5245 3484.3656 3233.7654 2328.8251 5423426.4490 314.2844  8418.5703  8104.2859    0.3013   -1.2057 1362.0469 5667.8247 4305.7778 66.8364
   Weekly  TLKM   1102 1850.0842 1770.8184 1036.7567 1074864.5519 347.5384  3898.2288  3550.6903    0.1544   -1.4579  830.4206 2740.5945 1910.1739 56.0384
   Weekly  BBCA   1102 3300.

## 3. Data Distribution Analysis

In [11]:
# ============================================================
# DISTRIBUTION PLOTS - Daily Data
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Close Price Distribution - Daily Data', fontsize=16, fontweight='bold', y=1.02)

for idx, stock in enumerate(STOCKS):
    ax = axes[idx // 2, idx % 2]
    close = daily_data[stock]['Close']
    
    ax.hist(close, bins=50, color=STOCK_COLORS[stock], alpha=0.7, 
            edgecolor='white', linewidth=0.5, density=True)
    
    # KDE overlay
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(close)
    x_range = np.linspace(close.min(), close.max(), 200)
    ax.plot(x_range, kde(x_range), color='black', linewidth=1.5, label='KDE')
    
    ax.set_title(f'{stock}', fontsize=14)
    ax.set_xlabel('Close Price (IDR)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.legend(fontsize=10)

fig.tight_layout()
save_fig(fig, 'figures/eda/distribution_daily_close.png')
print("Distribution plot saved.")

  Figure saved: figures/eda/distribution_daily_close.png
Distribution plot saved.


In [12]:
# ============================================================
# BOX PLOTS - Daily Data
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

box_data = [daily_data[stock]['Close'].values for stock in STOCKS]
bp = ax.boxplot(box_data, labels=STOCKS, patch_artist=True, 
                medianprops=dict(color='black', linewidth=2))

for patch, stock in zip(bp['boxes'], STOCKS):
    patch.set_facecolor(STOCK_COLORS[stock])
    patch.set_alpha(0.7)

ax.set_title('Close Price Box Plot - Daily Data', fontsize=16, fontweight='bold')
ax.set_xlabel('Stock', fontsize=14)
ax.set_ylabel('Close Price (IDR)', fontsize=14)

fig.tight_layout()
save_fig(fig, 'figures/eda/boxplot_daily_close.png')
print("Box plot saved.")

  Figure saved: figures/eda/boxplot_daily_close.png
Box plot saved.


In [13]:
# ============================================================
# VIOLIN PLOTS
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))

combined_df = pd.DataFrame()
for stock in STOCKS:
    temp = daily_data[stock][['Close']].copy()
    temp['Stock'] = stock
    combined_df = pd.concat([combined_df, temp], ignore_index=True)

parts = ax.violinplot([daily_data[s]['Close'].values for s in STOCKS],
                       positions=range(len(STOCKS)), showmeans=True, showmedians=True)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(STOCK_COLORS[STOCKS[i]])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(STOCKS)))
ax.set_xticklabels(STOCKS, fontsize=12)
ax.set_title('Close Price Violin Plot - Daily Data', fontsize=16, fontweight='bold')
ax.set_xlabel('Stock', fontsize=14)
ax.set_ylabel('Close Price (IDR)', fontsize=14)

fig.tight_layout()
save_fig(fig, 'figures/eda/violinplot_daily_close.png')
print("Violin plot saved.")

  Figure saved: figures/eda/violinplot_daily_close.png
Violin plot saved.


## 4. Normality Tests

In [14]:
# ============================================================
# NORMALITY TESTS
# ============================================================
normality_results = []
for stock in STOCKS:
    result = perform_normality_tests(daily_data[stock]['Close'].values, f'{stock}_Daily')
    normality_results.append(result)

norm_df = pd.DataFrame(normality_results)
print("\n" + "=" * 80)
print("  NORMALITY TESTS - Daily Close Prices")
print("=" * 80)
print(norm_df.to_string(index=False))

norm_df.to_csv('results/normality_tests.csv', index=False)
print("\nSaved to results/normality_tests.csv")


  NORMALITY TESTS - Daily Close Prices
      Name  Shapiro-Wilk Stat Shapiro-Wilk p-value Shapiro-Wilk Normal  KS Stat KS p-value KS Normal  DAgostino Stat DAgostino p-value DAgostino Normal
TLKM_Daily           0.902138             7.57e-49                  No 0.158073  6.72e-115        No    36993.104460          0.00e+00               No
BBCA_Daily           0.863031             1.19e-54                  No 0.174924  8.33e-141        No      731.122794         1.73e-159               No
ASII_Daily           0.864288             1.73e-54                  No 0.180443  7.19e-150        No     1217.838690         3.55e-265               No
UNVR_Daily           0.929382             1.55e-43                  No 0.096374   7.48e-43        No     4791.859352          0.00e+00               No

Saved to results/normality_tests.csv


In [15]:
# ============================================================
# Q-Q PLOTS
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Q-Q Plots - Daily Close Prices', fontsize=16, fontweight='bold', y=1.02)

for idx, stock in enumerate(STOCKS):
    ax = axes[idx // 2, idx % 2]
    stats.probplot(daily_data[stock]['Close'].values, dist="norm", plot=ax)
    ax.set_title(f'{stock}', fontsize=14)
    ax.get_lines()[0].set_color(STOCK_COLORS[stock])
    ax.get_lines()[0].set_markersize(3)
    ax.get_lines()[1].set_color('red')

fig.tight_layout()
save_fig(fig, 'figures/eda/qq_plots_daily.png')
print("Q-Q plots saved.")

  Figure saved: figures/eda/qq_plots_daily.png
Q-Q plots saved.


## 5. Stationarity Tests (Augmented Dickey-Fuller)

In [16]:
# ============================================================
# STATIONARITY TESTS (ADF)
# ============================================================
try:
    from statsmodels.tsa.stattools import adfuller
    
    adf_results = []
    for stock in STOCKS:
        # Test on raw prices
        result_raw = perform_stationarity_test(
            daily_data[stock]['Close'].values, f'{stock}_Raw'
        )
        adf_results.append(result_raw)
        
        # Test on returns (first difference)
        returns = daily_data[stock]['Close'].pct_change().dropna().values
        result_ret = perform_stationarity_test(returns, f'{stock}_Returns')
        adf_results.append(result_ret)
    
    adf_df = pd.DataFrame(adf_results)
    print("\n" + "=" * 80)
    print("  AUGMENTED DICKEY-FULLER TEST")
    print("=" * 80)
    print(adf_df.to_string(index=False))
    
    adf_df.to_csv('results/stationarity_tests.csv', index=False)
    print("\nSaved to results/stationarity_tests.csv")

except ImportError:
    print("statsmodels not installed. Install with: pip install statsmodels")
    print("Skipping ADF test.")


  AUGMENTED DICKEY-FULLER TEST
        Name  ADF Statistic  p-value Stationary (p<0.05)  Critical 1%  Critical 5%  Critical 10%
    TLKM_Raw      -0.897816 7.89e-01                  No      -3.4316      -2.8621       -2.5671
TLKM_Returns     -13.981824 4.16e-26                 Yes      -3.4316      -2.8621       -2.5671
    BBCA_Raw       0.061039 9.63e-01                  No      -3.4316      -2.8621       -2.5671
BBCA_Returns     -14.511077 5.72e-27                 Yes      -3.4316      -2.8621       -2.5671
    ASII_Raw      -1.160608 6.90e-01                  No      -3.4316      -2.8621       -2.5671
ASII_Returns     -19.079626 0.00e+00                 Yes      -3.4316      -2.8621       -2.5671
    UNVR_Raw      -1.399448 5.83e-01                  No      -3.4316      -2.8621       -2.5671
UNVR_Returns     -15.414059 3.10e-28                 Yes      -3.4316      -2.8621       -2.5671

Saved to results/stationarity_tests.csv


## 6. Correlation Analysis

In [17]:
# ============================================================
# CORRELATION MATRIX
# ============================================================
# Align all daily data by date
close_df = pd.DataFrame()
for stock in STOCKS:
    temp = daily_data[stock][['Date', 'Close']].copy()
    temp = temp.rename(columns={'Close': stock})
    if close_df.empty:
        close_df = temp
    else:
        close_df = close_df.merge(temp, on='Date', how='inner')

close_df = close_df.set_index('Date')
print(f"Aligned dataset: {close_df.shape[0]} common trading days\n")

# Pearson Correlation
corr_matrix = close_df.corr(method='pearson')
print("Pearson Correlation Matrix:")
print(corr_matrix.round(4))

# Spearman Correlation
spearman_corr = close_df.corr(method='spearman')
print("\nSpearman Correlation Matrix:")
print(spearman_corr.round(4))

# Save
corr_matrix.to_csv('results/pearson_correlation.csv')
spearman_corr.to_csv('results/spearman_correlation.csv')

Aligned dataset: 5243 common trading days

Pearson Correlation Matrix:
        TLKM    BBCA    ASII    UNVR
TLKM  1.0000  0.8685  0.8071  0.6439
BBCA  0.8685  1.0000  0.6698  0.3188
ASII  0.8071  0.6698  1.0000  0.7425
UNVR  0.6439  0.3188  0.7425  1.0000

Spearman Correlation Matrix:
        TLKM    BBCA    ASII    UNVR
TLKM  1.0000  0.9182  0.8475  0.6796
BBCA  0.9182  1.0000  0.8015  0.6047
ASII  0.8475  0.8015  1.0000  0.7384
UNVR  0.6796  0.6047  0.7384  1.0000


In [18]:
# Correlation Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pearson
sns.heatmap(corr_matrix, annot=True, fmt='.4f', cmap='RdYlBu_r',
            ax=axes[0], vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 12})
axes[0].set_title('Pearson Correlation', fontsize=14)
axes[0].tick_params(labelsize=12)

# Spearman
sns.heatmap(spearman_corr, annot=True, fmt='.4f', cmap='RdYlBu_r',
            ax=axes[1], vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 12})
axes[1].set_title('Spearman Correlation', fontsize=14)
axes[1].tick_params(labelsize=12)

fig.suptitle('Stock Price Correlation Analysis', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'figures/eda/correlation_heatmaps.png')
print("Correlation heatmaps saved.")

  Figure saved: figures/eda/correlation_heatmaps.png
Correlation heatmaps saved.


In [19]:
# Pairwise scatter plots
fig = plt.figure(figsize=(14, 14))
from itertools import combinations

pairs = list(combinations(STOCKS, 2))
n_pairs = len(pairs)
n_cols = 3
n_rows = (n_pairs + n_cols - 1) // n_cols

for idx, (s1, s2) in enumerate(pairs):
    ax = fig.add_subplot(n_rows, n_cols, idx + 1)
    ax.scatter(close_df[s1], close_df[s2], alpha=0.3, s=5, color='#0072B2')
    ax.set_xlabel(s1, fontsize=11)
    ax.set_ylabel(s2, fontsize=11)
    
    # Add Pearson r
    r = corr_matrix.loc[s1, s2]
    ax.text(0.05, 0.95, f'r = {r:.4f}', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Pairwise Scatter Plots - Daily Close Prices', 
             fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, 'figures/eda/pairwise_scatter.png')
print("Pairwise scatter plots saved.")

  Figure saved: figures/eda/pairwise_scatter.png
Pairwise scatter plots saved.


## 7. Time Series Visualization

In [20]:
# ============================================================
# TIME SERIES PLOTS - ALL STOCKS
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

for stock in STOCKS:
    ax.plot(daily_data[stock]['Date'], daily_data[stock]['Close'],
            color=STOCK_COLORS[stock], label=stock, linewidth=1.2, alpha=0.9)

ax.set_title('Daily Close Prices - All Stocks', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Close Price (IDR)', fontsize=14)
ax.legend(fontsize=12, loc='best')
ax.tick_params(axis='x', rotation=30)

fig.tight_layout()
save_fig(fig, 'figures/eda/time_series_all_stocks.png')
print("Time series plot saved.")

  Figure saved: figures/eda/time_series_all_stocks.png
Time series plot saved.


In [21]:
# Individual stock time series with all timeframes
for stock in STOCKS:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'{stock} - Close Price Across Timeframes', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    for idx, (tf_name, tf_dict) in enumerate([
        ('Daily', daily_data), ('Weekly', weekly_data),
        ('Monthly', monthly_data), ('Yearly', yearly_data)
    ]):
        ax = axes[idx // 2, idx % 2]
        df = tf_dict[stock]
        ax.plot(df['Date'], df['Close'], color=STOCK_COLORS[stock], linewidth=1.2)
        ax.set_title(f'{tf_name} ({len(df)} points)', fontsize=13)
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylabel('Close Price (IDR)', fontsize=11)
        ax.tick_params(axis='x', rotation=30)
    
    fig.tight_layout()
    save_fig(fig, f'figures/eda/{stock}_timeframes.png')

print("Individual timeframe plots saved.")

  Figure saved: figures/eda/TLKM_timeframes.png
  Figure saved: figures/eda/BBCA_timeframes.png
  Figure saved: figures/eda/ASII_timeframes.png
  Figure saved: figures/eda/UNVR_timeframes.png
Individual timeframe plots saved.


In [22]:
# ============================================================
# SCALED DATA VISUALIZATION (ProportionScaler)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))

for stock in STOCKS:
    scaled = proportion_scale(daily_data[stock]['Close'].values)
    ax.plot(daily_data[stock]['Date'], scaled,
            color=STOCK_COLORS[stock], label=stock, linewidth=1.2, alpha=0.9)

ax.set_title('ProportionScaler Applied (÷ 10,501) - Daily Close', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Scaled Close Price', fontsize=14)
ax.legend(fontsize=12, loc='best')
ax.tick_params(axis='x', rotation=30)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5, label='Max (BBCA ATH)')

fig.tight_layout()
save_fig(fig, 'figures/eda/scaled_time_series_all_stocks.png')
print("Scaled time series plot saved.")

  Figure saved: figures/eda/scaled_time_series_all_stocks.png
Scaled time series plot saved.


## 8. Returns Analysis

In [23]:
# ============================================================
# DAILY RETURNS ANALYSIS
# ============================================================
returns_data = {}
for stock in STOCKS:
    returns_data[stock] = daily_data[stock]['Close'].pct_change().dropna()

# Returns distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Daily Returns Distribution', fontsize=16, fontweight='bold', y=1.02)

for idx, stock in enumerate(STOCKS):
    ax = axes[idx // 2, idx % 2]
    ret = returns_data[stock]
    
    ax.hist(ret, bins=80, color=STOCK_COLORS[stock], alpha=0.7, 
            edgecolor='white', linewidth=0.3, density=True)
    
    # Normal distribution overlay
    mu, sigma = ret.mean(), ret.std()
    x = np.linspace(ret.min(), ret.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=1.5, label='Normal')
    
    ax.set_title(f'{stock} (μ={mu:.5f}, σ={sigma:.4f})', fontsize=13)
    ax.set_xlabel('Daily Return', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=10)

fig.tight_layout()
save_fig(fig, 'figures/eda/returns_distribution.png')

# Returns statistics
ret_stats = []
for stock in STOCKS:
    ret = returns_data[stock]
    ret_stats.append({
        'Stock': stock,
        'Mean Return': f'{ret.mean():.6f}',
        'Std Return': f'{ret.std():.6f}',
        'Min Return': f'{ret.min():.6f}',
        'Max Return': f'{ret.max():.6f}',
        'Skewness': f'{ret.skew():.4f}',
        'Kurtosis': f'{ret.kurtosis():.4f}',
    })

ret_stats_df = pd.DataFrame(ret_stats)
print("\nDaily Returns Statistics:")
print(ret_stats_df.to_string(index=False))
ret_stats_df.to_csv('results/returns_statistics.csv', index=False)
print("\nReturns analysis saved.")

  Figure saved: figures/eda/returns_distribution.png

Daily Returns Statistics:
Stock Mean Return Std Return Min Return Max Return Skewness Kurtosis
 TLKM    0.000634   0.019597  -0.099237   0.174242   0.3960   4.0400
 BBCA    0.000967   0.018580  -0.100775   0.173334   0.3176   5.3802
 ASII    0.000877   0.023160  -0.200000   0.198718   0.4111   6.1458
 UNVR    0.000620   0.021421  -0.154167   0.196079   0.7944   8.2482

Returns analysis saved.


In [24]:
# ============================================================
# ROLLING STATISTICS
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Rolling Mean & Standard Deviation (60-day window)', 
             fontsize=16, fontweight='bold', y=1.02)

for idx, stock in enumerate(STOCKS):
    ax = axes[idx // 2, idx % 2]
    close = daily_data[stock].set_index('Date')['Close']
    
    rolling_mean = close.rolling(window=60).mean()
    rolling_std = close.rolling(window=60).std()
    
    ax.plot(close.index, close, color=STOCK_COLORS[stock], alpha=0.4, 
            linewidth=0.8, label='Close')
    ax.plot(close.index, rolling_mean, color='black', linewidth=1.5, 
            label='60-day Mean')
    ax.fill_between(close.index, rolling_mean - 2*rolling_std, 
                    rolling_mean + 2*rolling_std, alpha=0.15, color='gray',
                    label='±2σ Band')
    
    ax.set_title(f'{stock}', fontsize=13)
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Close Price (IDR)', fontsize=11)
    ax.legend(fontsize=9, loc='best')
    ax.tick_params(axis='x', rotation=30)

fig.tight_layout()
save_fig(fig, 'figures/eda/rolling_statistics.png')
print("Rolling statistics plot saved.")

  Figure saved: figures/eda/rolling_statistics.png
Rolling statistics plot saved.


## 9. Summary

All EDA results have been saved to:
- `results/` - CSV files with statistics
- `figures/eda/` - Publication-quality plots (600 DPI)

**Key files generated:**
- `descriptive_statistics_daily.csv`
- `descriptive_statistics_all_timeframes.csv`
- `normality_tests.csv`
- `stationarity_tests.csv`
- `pearson_correlation.csv`
- `spearman_correlation.csv`
- `returns_statistics.csv`

Proceed to Notebook 1 for model training experiments.
